# ChatGPT initial prompt
### "I have a text dataset of tweets. I need to use regular expressions to analyze patterns in the tweets, such as dates, words starting with specific letters, emojis, IP addresses, hashtags, URLs, and more. Also, I need to perform stemming and lemmatization, compare the word counts, and identify changes in word frequencies. Can you guide me step by step and provide code examples for each task?"

# **MECE Table**

|   Task No. | Task Description                                                   | Regex/NLP Applied                 | Output Metric                      | Assigned To                  |
|-----------:|:-------------------------------------------------------------------|:----------------------------------|:-----------------------------------|:-----------------------------|
|          1 | Records with dates expressed without alphabets                     | \b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b | Count of matching records          | Om Kiranbhai Patel           |
|          2 | Records with words starting with 'w'                               | \bw\w*                            | Count of records with at least one | Om Kiranbhai Patel           |
|          3 | Words starting with alphabets but not URLs                         | \b(?!http,www)[a-zA-Z]\w*\b       | Count of valid words               | Tanzima Mohammadyasin Shaikh |
|          4 | Records containing emojis :), :D, ;), :P                           | :\),:D,;\),:P                     | Count of matching records          | Tanzima Mohammadyasin Shaikh |
|          5 | Records with decimal numbers                                       | \b\d+\.\d+\b                      | Count of records with decimals     | Arish Panjwani               |
|          6 | Total IP addresses across all records                              | \b\d{1,3}(?:\.\d{1,3}){3}\b       | Total number of IPs found          | Arish Panjwani               |
|          7 | Records containing a new line                                      | \n or multiline check             | Count of records                   | Nischal Pradhan              |
|          8 | Total hashtags across all tweets                                   | #\w+                              | Total number of hashtags           | Nischal Pradhan              |
|          9 | Replace all non-alphanumeric characters with newline               | [^a-zA-Z0-9]+ → \n                | Transformed output (code only)     | Advait Manishkumar Pandit    |
|         10 | Total number of URLs in tweets                                     | https?://\S+,www\.\S+             | Total number of URLs               | Advait Manishkumar Pandit    |
|         11 | Stemming using Porter – unique words before & after                | NLTK PorterStemmer                | Unique word counts                 | Mueez Ur Rehman Amjad        |
|         12 | Lemmatization using WordNet – unique words before & after          | NLTK WordNetLemmatizer            | Unique word counts                 | Kanika .                     |
|         13 | Top 10 words after stemming vs lemmatization                       | Word frequency count              | Top 10 comparison                  | Riya Kalpeshkumar Shah       |
|         14 | Change in word frequencies after stop word removal + normalization | Lowercasing + removing stop words | Frequency comparison               | Ashish Lama                  |


In [40]:
%pip install pandas nltk

Note: you may need to restart the kernel to use updated packages.


In [41]:
import pandas as pd
import re
import nltk
import string
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from collections import Counter
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/tanzimashaikh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/tanzimashaikh/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/tanzimashaikh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/tanzimashaikh/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [42]:
df = pd.read_csv('Data.csv')

In [43]:
# Display the first few rows
df.head()

,Data
0,Watch or listen live weekdays at 8:30am MT at ...
1,Watch or listen live weekdays at 8:30am MT at ...
2,"Chubby And Hot, Always Stir The Pot!"
3,"Chubby And Hot, Always Stir The Pot!"
4,"Journalist, publisher of Rebel News — telling ..."


In [44]:
# Display last few rows
df.tail()

,Data
7334,Snow Valley Racing delivers competitive ski ra...
7335,New twin mom\n\nMorning News Anchor @citynewsc...
7336,Residents advocating for the💚of the Park.\n\nP...
7337,"Driven by family, diversity, equity, inclusion..."
7338,"Photographer, Oilers Fan #photography #yeg ."


### 1. Records with date without alphabets

In [45]:
# Defining the regex pattern for numeric dates
# For example, 12-05-2023, 2023/01/01, 12.05.23, 2023-01-01, 01/12/2022
regex_date_pattern = r'\b\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}\b'

In [46]:
# Check each row in the column to see if it contains such a date
df['num_date'] = df['Data'].apply(lambda x: bool(re.search(regex_date_pattern, str(x))))

In [47]:
# Count how many rows have dates written without any letters
numeric_dates = df['num_date'].sum()

In [48]:
# Display the result
print("Number of records with dates expressed without using alphabets:", numeric_dates)

Number of records with dates expressed without using alphabets: 4


### 2. Records with a word starting with “w”

In [49]:
# Define the regex pattern for words starting with 'w' or 'W'
wpattern = r'\b[wW]\w*'

In [50]:
# Check each row to see if it has a word that starts with 'w'
df['has_word_starting_with_w'] = df['Data'].apply(lambda x: bool(re.search(wpattern, str(x))))

In [51]:
# Count how many rows have such words
words_starting_with_w = df['has_word_starting_with_w'].sum()

In [52]:
# Display the result
print("Number of records with a word starting with the letter 'w':", words_starting_with_w)

Number of records with a word starting with the letter 'w': 3591


### 3. Words starting with alphabets but not URLs

In [53]:
# Function to check for a word starting with a letter but not a URL
def contains_valid_word_regex(text):
    if isinstance(text, str):
        # This regex finds words that start with a letter
        words = re.findall(r'\b[a-zA-Z][\w\-]*', text)
        for word in words:
            # Excluding links (http, https, www)
            if not re.match(r'^(http|https|www)', word.lower()):
                return True
    return False
#  Count how many tweets match
q3_count = df['Data'].apply(contains_valid_word_regex).sum()
# Result
print("Q3: Number of tweets with a word starting with an alphabet (not a URL):", q3_count)

Q3: Number of tweets with a word starting with an alphabet (not a URL): 7305


### 4. Records containing emojis :), :D, ;), :P

In [54]:
# Regex pattern for emojis: :) :D ;) :P
emoji_pattern = r"(?:\:\)|:D|;\)|:P)"
q4_count = df['Data'].str.contains(emoji_pattern, regex=True, na=False).sum()
# Result
print("Q4: Number of tweets containing one of the emojis :) :D ;) :P :", q4_count)


Q4: Number of tweets containing one of the emojis :) :D ;) :P : 18


### 5. Records with a decimal number

In [55]:
df['decimal'] = df['Data'].str.contains(r'\d+\.\d+')
print('There are',df['decimal'].sum(),'records with a decimal number')

There are 27 records with a decimal number


### 6. Total IP addresses across all records

In [56]:
pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
df['ip_count'] = df['Data'].str.findall(pattern).apply(len)
df['ip_count'].sum()
print('There are total',df['ip_count'].sum(),'IP addresses across all records')

There are total 0 IP addresses across all records


### 7. Records containing a new line

In [57]:
# Q7. Count of records containing newline characters
newline_count = df['Data'].astype(str).str.contains(r'\n', regex=True).sum()

In [58]:
print("Number of records containing newline characters:", newline_count)

Number of records containing newline characters: 1211


### 8 Total hashtags across all tweets

In [59]:
# extracting hashtags from the Data, using regex
def extract_hashtags(x):
    return re.findall(r'#\w+', x)

# applying extraction to all hashtags from Data
df['hashtags'] = df['Data'].astype(str).apply(extract_hashtags)

# counting the number of hashtags in each tweet
df['hashtag_count'] = df['hashtags'].apply(len)

# printing total number of hashtags
total_hashtags_counts = df['hashtag_count'].sum()
print("Total number of hashtags:", total_hashtags_counts)

Total number of hashtags: 2924


### 9 Replace all non-alphanumeric characters with newline

In [60]:
#Q-9: What is the code to substitute all non-alphanumeric characters with a new line?
import re

df['cleaned_text'] = df['Data'].astype(str).apply(lambda x: re.sub(r'[^a-zA-Z0-9]', '\n', x))

# Printing one example
print("Example cleaned output for one record:\n")
print(df['cleaned_text'].iloc[0])

Example cleaned output for one record:

Watch
or
listen
live
weekdays
at
8
30am
MT
at
ryanjespersen
com

Subscribe
via
YouTube
or
your
favourite
podcast
app


RealTalkRJ


### 10 Total number of URLs in tweets

In [61]:
# Q-10: What is the total number of URLs across all tweets?
import re
import pandas as pd


texts = df['Data'].astype(str)

# Regex
url = re.compile(r'https?://\S+|www\.\S+')


total_urls = texts.str.count(url).sum()
print("Total URLs found across all records:", total_urls)


Total URLs found across all records: 4


### 11 Stemming using Porter – unique words before & after

In [62]:
# Combine all text into one lowercase string
all_text = " ".join(df['Data'].dropna().astype(str).tolist()).lower()


import re
# Extracting only words
tokens = re.findall(r'\b[a-zA-Z]+\b', all_text)

# Count unique words before stemming
unique_before = set(tokens)

from nltk.stem import PorterStemmer

# Apply Porter Stemmer
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(token) for token in tokens]

# Count unique words after stemming
unique_after = set(stemmed_tokens)

# Print the results
print(f"Unique words before stemming: {len(unique_before)}")
print(f"Unique words after stemming: {len(unique_after)}")


Unique words before stemming: 7445
Unique words after stemming: 6088


### 12 Lemmatization using WordNet – unique words before & after

In [63]:
text_data = df.iloc[:, 0].dropna().astype(str).tolist()

# Combine all text and clean
full_text = " ".join(text_data).lower()
full_text = full_text.translate(str.maketrans('', '', string.punctuation))

# Tokenize
tokens = nltk.word_tokenize(full_text)

# Unique tokens before lemmatization
unique_before = set(tokens)

# Lemmatize
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
unique_after = set(lemmatized_tokens)

# Results
print(f"Total tokens: {len(tokens)}")
print(f"Unique tokens before lemmatization: {len(unique_before)}")
print(f"Unique tokens after lemmatization: {len(unique_after)}")

print("\nSample unique words before:", list(unique_before)[:10])
print("Sample unique words after:", list(unique_after)[:10])

Total tokens: 117689
Unique tokens before lemmatization: 8714
Unique tokens after lemmatization: 8155

Sample unique words before: ['ccis', 'amberdennisonmstdnca', 'guest', 'mcc', '🚜', 'tinkerer', 'pandemic', 'imperfectly', 'punk', 'eat']
Sample unique words after: ['ccis', 'amberdennisonmstdnca', 'guest', 'mcc', '🚜', 'tinkerer', 'pandemic', 'imperfectly', 'punk', 'eat']


### 13 Top 10 words after stemming vs lemmatization

In [64]:
stop_words = set(stopwords.words('english'))  

def clean_text(text):
    try:
        tokens = text.lower().split()
        tokens = [word for word in tokens if word not in stop_words and word.isalpha()]
        return tokens
    except Exception as e:
        print(f"Error processing text: {text} -> {e}")
        return []
df['Cleaned_Data'] = df['Data'].apply(clean_text)

all_tokens = [word for tokens in df['Cleaned_Data'] for word in tokens]

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = [stemmer.stem(word) for word in all_tokens]
lemmatized_words = [lemmatizer.lemmatize(word) for word in all_tokens]

stem_freq = Counter(stemmed_words).most_common(10)
lemma_freq = Counter(lemmatized_words).most_common(10)

print("Top 10 after stemming:", stem_freq)
print("Top 10 after lemmatization:", lemma_freq)

comparison = pd.DataFrame({
    'Stemmed': dict(stem_freq),
    'Lemmatized': dict(lemma_freq)
}).fillna(0).astype(int)

print(comparison)

Top 10 after stemming: [('alberta', 644), ('outdoor', 635), ('adventur', 492), ('commun', 467), ('news', 412), ('love', 392), ('follow', 386), ('natur', 320), ('canada', 306), ('travel', 272)]
Top 10 after lemmatization: [('alberta', 644), ('outdoor', 545), ('adventure', 488), ('community', 439), ('news', 412), ('follow', 353), ('love', 345), ('canada', 306), ('park', 236), ('official', 220)]
           Stemmed  Lemmatized
alberta        644         644
outdoor        635         545
adventur       492           0
commun         467           0
news           412         412
love           392         345
follow         386         353
natur          320           0
canada         306         306
travel         272           0
adventure        0         488
community        0         439
park             0         236
official         0         220


### 14 Change in word frequencies after stop word removal + normalization

##### Setting stop words

In [65]:
stop_words = set(stopwords.words('english'))

##### Function to filter and tokenize words

In [66]:
def tokenize_and_filter(text, normalize=False):
    tokens = nltk.word_tokenize(text)
    filtered_words = [word for word in tokens if word.lower() not in stop_words]
    if normalize:
        filtered_words = [word.lower() for word in filtered_words]
    return filtered_words

##### Filtering the data frame based with and without normalization

In [67]:
df['filtered_df_with_no_norm'] = df['Data'].apply(lambda x: tokenize_and_filter(str(x), normalize=False))
df['filtered_df_with_norm'] = df['Data'].apply(lambda x: tokenize_and_filter(str(x), normalize=True))

In [68]:
df['filtered_df_with_no_norm'].head()

0    [Watch, listen, live, weekdays, 8:30am, MT, ry...
1    [Watch, listen, live, weekdays, 8:30am, MT, ry...
2               [Chubby, Hot, ,, Always, Stir, Pot, !]
3               [Chubby, Hot, ,, Always, Stir, Pot, !]
4    [Journalist, ,, publisher, Rebel, News, —, tel...
Name: filtered_df_with_no_norm, dtype: object

In [69]:
df['filtered_df_with_norm'].head()

0    [watch, listen, live, weekdays, 8:30am, mt, ry...
1    [watch, listen, live, weekdays, 8:30am, mt, ry...
2               [chubby, hot, ,, always, stir, pot, !]
3               [chubby, hot, ,, always, stir, pot, !]
4    [journalist, ,, publisher, rebel, news, —, tel...
Name: filtered_df_with_norm, dtype: object

##### Getting the list of all words

In [70]:
all_words_with_no_norm = [word for tokens in df['filtered_df_with_no_norm'] for word in tokens]
all_words_with_norm = [word for tokens in df['filtered_df_with_norm'] for word in tokens]

##### Frequency count

In [71]:
freq_with_no_norm = Counter(all_words_with_no_norm)
freq_with_norm = Counter(all_words_with_norm)

##### Converting the frequency count to Dataframe for comparison

In [72]:
df_freq = pd.DataFrame.from_dict(freq_with_no_norm, orient='index', columns=['No_Normalization'])
df_freq['With_Normalization'] = pd.Series(freq_with_norm)
df_freq.fillna(0, inplace=True)

#### Creating a new column Change in the DataFrame

In [73]:
df_freq['Change'] = df_freq['With_Normalization'] - df_freq['No_Normalization']

#### Top 20 words with normalization and no normalization

In [74]:
df_freq = df_freq.sort_values(by='Change', ascending=False)
df_freq.head(20)

,No_Normalization,With_Normalization,Change
alberta,6,1059.0,1053.0
outdoor,98,549.0,451.0
calgary,14,390.0,376.0
writer,54,383.0,329.0
follow,120,361.0,241.0
canadian,6,222.0,216.0
adventures,179,373.0,194.0
edmonton,8,201.0,193.0
twitter,44,230.0,186.0
speaker,4,181.0,177.0


# ChatGPT Last prompt
### "I’ve implemented all the required regex-based text analysis, stemming, lemmatization, and word frequency comparisons. Could you review the code for clarity and completeness, and suggest any improvements or best practices to finalize the notebook for submission?"